# NumPy for Deep Learning: companion notebook

Experiments for Appendix B. Run the cells in order, then change the numbers and run them again. Everything runs in your browser; `scratch` is the book's shared NumPy package.

In [ ]:
import numpy as np

from scratch.arrays import unbroadcast
from scratch.gradcheck import check_gradient, numerical_gradient, relative_error
from scratch.precision import round_to_bfloat16

rng = np.random.default_rng(0)
print('NumPy', np.__version__)

## 1. Predict a broadcast, then check

Write down each result shape before running the cell. `np.broadcast_shapes` answers without allocating arrays.

In [ ]:
pairs = [((4, 1, 3), (5, 1)), ((4, 1, 3), (3,)), ((5, 1), (3,)), ((4, 1, 3), (4, 3)), ((4, 3), (5, 1))]
for a, b in pairs:
    try:
        print(f'{a} + {b} -> {np.broadcast_shapes(a, b)}')
    except ValueError:
        print(f'{a} + {b} -> error')

## 2. The gradient of a broadcast is a sum

For $Y = X + b$, the loss $L = \sum_{ij} G_{ij} Y_{ij}$ has $\bar{b} = \sum_i G_{i,:}$. `unbroadcast` computes it; `check_gradient` compares it with central differences.

In [ ]:
X, G = rng.standard_normal((6, 4)), rng.standard_normal((6, 4))
b = rng.standard_normal(4)
grad_b = unbroadcast(G, b.shape)
error = check_gradient(lambda v: float(np.sum(G * (X + v))), b, grad_b)
print('bias gradient', np.round(grad_b, 4), f'relative error {error:.1e}')

## 3. Log-sum-exp without overflow

$\operatorname{logsumexp}(z) = m + \log \sum_i e^{z_i - m}$ with $m = \max_i z_i$.

In [ ]:
z = np.array([1000.0, 1001.0, 1002.0], dtype=np.float32)
with np.errstate(over='ignore'):
    naive = np.log(np.sum(np.exp(z)))
m = z.max()
stable = m + np.log(np.sum(np.exp(z - m)))
print('naive', naive, ' stable', stable)

## 4. A running sum that stalls

Add 0.001 ten thousand times, rounding after every step. Try other addends and step counts.

In [ ]:
def running_sum(value, steps, round_to):
    total = round_to(np.float32(0))
    for _ in range(steps):
        total = round_to(total + round_to(np.float32(value)))
    return float(total)

for name, round_to in [('bfloat16', round_to_bfloat16), ('float16', np.float16), ('float32', np.float32)]:
    print(f'{name:>8}: {running_sum(1e-3, 10_000, round_to):.4f}')

## 5. Your turn: check a gradient

The derivative of $\tanh$ is $1 - \tanh^2$. Replace `analytic` with a wrong formula and watch the check fail.

In [ ]:
x = rng.standard_normal(5)
analytic = 1 - np.tanh(x) ** 2
error = check_gradient(lambda v: float(np.sum(np.tanh(v))), x, analytic)
print(f'tanh gradient check passed: relative error {error:.1e}')